In [22]:
import sys
import os
import scMPRAforge as scm
import pandas as pd
import numpy as np

#load the autoreload extension
%load_ext autoreload
#reload code on every execution
#(this may break objects)
#you can remove this & do dev in a notebook, then paste into the module when you are done.
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [1]:
from dask.distributed import Client, LocalCluster
cluster=LocalCluster(memory_limit='8GB')
client=Client(cluster)

In [2]:
path="/gpfs/gibbs/pi/reilly/tabula_data/shendure"
name="ortho_primordial_v3"
data_root="/gpfs/gibbs/pi/reilly/tabula_data"
primordial=scm.ortho.load(client,path,name)

In [23]:
scm.SHENDURE_BOUNDS#.cells_per_cell_type

Bounds(metadata=None, preferred='by_cell_type', min_mpra_umi=0.0034641979561541277, max_mpra_umi=369.88473782763344, by_cre_theta=0.62110305, by_cell_type_theta=0.35434753, zi=0.022971882287718185, by_cre_zi=0.39563528657381514, by_cell_type_zi=0.022971882287718185, num_cres=208, cells_per_cell_type=cell_type
Cardiomyocytes              680
EpiblastPrimitiveStreak    3445
ExEndodermParietal         4644
ExEndodermVisceral         3238
Haematoendothelial         1079
Mesoderm                   7427
NeuroectodermBrain         7750
NeuroectodermRostral       1757
SurfaceEctoderm            5168
reference                  8201
Name: cells_per_cell_type, dtype: int64, transfection_nb_mu=18.010555670792133, transfection_nb_alpha=0.5705626300443596)

Here's an example of creating an artificial experiment.

In [9]:
#first, we define the new parameters we want to assign to this object.
new_cell_number=pd.Series({"reference":1000,"blood":2000,"neuron":1000})

new_min=1
new_max=200
new_zi=0.05

new_MOI=30

In [10]:
#next, let's create 
artificial_bounds=scm.SHENDURE_BOUNDS.copy(
    min_mpra_umi=new_min,
    max_mpra_umi=new_max,
    zi=new_zi)
artificial_bounds.set_effective_moi(new_MOI)

In [11]:
spread=scm.simple_spread(cell_types=new_cell_number.keys(),
                  min=new_min,
                  max=new_max)

In [15]:
from scipy.stats import nbinom

In [ ]:
def description_from_bounds(experiment_bounds:scm.Bounds,
                            spread:pd.DataFrame):
    """
    Returns a primordial description dask dataframe from bounds
    and ground truth dataframe. 

    See README spec for details on ground truth dataframe.
    You can easially create one with the helper function `simple_spread`. 
    """

    #known before you start or "to be optimized":
    #  cells per cell-type is a fixed parameter
    #  barcodes per CRE is a fixed parameter

    #for each cell type, cell, decide how many MPRA barcodes are transfected.
    #note that MOI here is measured, not applied MOI. Transfection inefficiencies
    #Will effectively reduce MOI from what is applied. 
    #probably the way to do it is
    # - calculate fraction of cells with 0, 1, 2, 3... barcodes
    # - multiple fractions by total number of cells & round
    # - make a table with n=total_cells cell barcodes
    # - add number of barcodes in each cell as a column according to fraction (n_transfected).
    # - randomize row order
    # - assign cell_type based on proportions
    # - randomly sample n_transfected MPRA barcodes for each row, then convert to tall
    # - merge in CRE identity 

    #for a non-tfection reporter setup, extend to 'all-by-all' (all mpra BC by all) filling in zeroes

    #The output will look like an ortho describe_primordial description dataframe. 

    ## calculate fraction of cells with 0, 1, 2, 3... n barcodes ##

    #make sure r & p are up to date!
    experiment_bounds.update_transfection_params()
    
    total_cells=experiment_bounds.cells_per_cell_type.sum()

    #how much probability density of the transfection model do we want
    #we want density out to y where, pdf*total cells=1
    #e.g. we miss less than 1 cell total. 
    required_density=1-1/total_cells
    
    k_max = int(np.ceil(nbinom.ppf(required_density, experiment_bounds.r, experiment_bounds.p)))
    k = np.arange(0, k_max + 1)
    pmf_transfection = nbinom.pmf(k, experiment_bounds.r, experiment_bounds.p)

    transfection=np.round(pmf_transfection*total_cells)
    
    #transfection[0]=n cells with 0 MPRA barcode transfected,
    #transfection[1]=n cells with 1 MPRA barcode transfected
    #etc...

    ## make a table of cell barcodes ##
    
    
    pass


In [22]:
description_from_bounds(experiment_bounds=artificial_bounds,
                        spread=spread)

[2.710e+02 4.480e+02 5.830e+02 6.890e+02 7.730e+02 8.400e+02 8.940e+02
 9.350e+02 9.670e+02 9.900e+02 1.005e+03 1.015e+03 1.019e+03 1.019e+03
 1.014e+03 1.006e+03 9.950e+02 9.820e+02 9.670e+02 9.490e+02 9.310e+02
 9.110e+02 8.900e+02 8.680e+02 8.460e+02 8.240e+02 8.010e+02 7.780e+02
 7.540e+02 7.310e+02 7.080e+02 6.850e+02 6.630e+02 6.400e+02 6.190e+02
 5.970e+02 5.760e+02 5.550e+02 5.350e+02 5.150e+02 4.960e+02 4.770e+02
 4.590e+02 4.410e+02 4.240e+02 4.070e+02 3.910e+02 3.750e+02 3.600e+02
 3.450e+02 3.310e+02 3.180e+02 3.040e+02 2.920e+02 2.790e+02 2.680e+02
 2.560e+02 2.450e+02 2.350e+02 2.250e+02 2.150e+02 2.060e+02 1.970e+02
 1.880e+02 1.800e+02 1.720e+02 1.640e+02 1.570e+02 1.500e+02 1.430e+02
 1.370e+02 1.300e+02 1.250e+02 1.190e+02 1.130e+02 1.080e+02 1.030e+02
 9.900e+01 9.400e+01 9.000e+01 8.600e+01 8.200e+01 7.800e+01 7.400e+01
 7.100e+01 6.700e+01 6.400e+01 6.100e+01 5.800e+01 5.600e+01 5.300e+01
 5.000e+01 4.800e+01 4.600e+01 4.400e+01 4.100e+01 3.900e+01 3.800e+01
 3.600

In [ ]:
#we want to compute "fraction cells with n barcodes" out to
#the point where fraction * total cells <1 cell.
#we can express in terms of percent density
#what percent 

artificial_bounds.cells_per_cell_type.sum()

43389

In [10]:
cluster.close()